In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Imports loaded.")
print("Loading application_train...")
start = datetime.now()

app = pd.read_csv('../data/raw/application_train.csv')

# Re-apply the DAYS_EMPLOYED fix from EDA
app['DAYS_EMPLOYED_ANOM'] = (app['DAYS_EMPLOYED'] == 365243).astype(int)
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"application_train loaded: {app.shape}")
print(f"Time: {datetime.now() - start}")
print(f"\nNote: working only with application_train here.")
print(f"Subsidiary tables loaded on demand to manage memory.")

Imports loaded.
Loading application_train...
application_train loaded: (307511, 123)
Time: 0:00:02.615428

Note: working only with application_train here.
Subsidiary tables loaded on demand to manage memory.


In [2]:
print("Engineering features from application_train...")

# Age and employment in human-readable units
app['AGE_YEARS'] = app['DAYS_BIRTH'] / -365
app['YEARS_EMPLOYED'] = app['DAYS_EMPLOYED'] / -365

# Core credit risk ratios
app['CREDIT_INCOME_RATIO'] = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']
app['ANNUITY_INCOME_RATIO'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']
app['CREDIT_TERM'] = app['AMT_ANNUITY'] / app['AMT_CREDIT']

# EXT_SOURCE combined score — weighted toward strongest predictor
app['EXT_SOURCE_MEAN'] = app[['EXT_SOURCE_1',
                               'EXT_SOURCE_2',
                               'EXT_SOURCE_3']].mean(axis=1)
app['EXT_SOURCE_WEIGHTED'] = (
    app['EXT_SOURCE_1'].fillna(app['EXT_SOURCE_MEAN']) * 0.30 +
    app['EXT_SOURCE_2'].fillna(app['EXT_SOURCE_MEAN']) * 0.32 +
    app['EXT_SOURCE_3'].fillna(app['EXT_SOURCE_MEAN']) * 0.38
)

# Income per family member
app['INCOME_PER_PERSON'] = (
    app['AMT_INCOME_TOTAL'] / app['CNT_FAM_MEMBERS']
)

# Document flags — how many documents did applicant provide?
doc_cols = [c for c in app.columns if 'FLAG_DOCUMENT' in c]
app['DOCUMENT_COUNT'] = app[doc_cols].sum(axis=1)

# Contact reachability — can lender reach this person?
app['CONTACT_REACHABILITY'] = (
    app['FLAG_MOBIL'] +
    app['FLAG_EMP_PHONE'] +
    app['FLAG_WORK_PHONE'] +
    app['FLAG_PHONE'] +
    app['FLAG_EMAIL']
)

# Verify new features
new_features = [
    'AGE_YEARS', 'YEARS_EMPLOYED', 'CREDIT_INCOME_RATIO',
    'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'EXT_SOURCE_MEAN',
    'EXT_SOURCE_WEIGHTED', 'INCOME_PER_PERSON',
    'DOCUMENT_COUNT', 'CONTACT_REACHABILITY'
]

print(f"\nNew features created: {len(new_features)}")
print(f"app shape after engineering: {app.shape}")
print(f"\nSample values (first row):")
for f in new_features:
    print(f"  {f:<30} {app[f].iloc[0]:.4f}")

# Quick correlation check against TARGET
print(f"\nCorrelation with TARGET:")
for f in new_features:
    corr = app[f].corr(app['TARGET'])
    direction = "↑ higher = more risk" if corr > 0 else "↓ higher = less risk"
    print(f"  {f:<30} {corr:+.4f}  {direction}")

Engineering features from application_train...

New features created: 10
app shape after engineering: (307511, 133)

Sample values (first row):
  AGE_YEARS                      25.9205
  YEARS_EMPLOYED                 1.7452
  CREDIT_INCOME_RATIO            2.0079
  ANNUITY_INCOME_RATIO           0.1220
  CREDIT_TERM                    0.0607
  EXT_SOURCE_MEAN                0.1618
  EXT_SOURCE_WEIGHTED            0.1620
  INCOME_PER_PERSON              202500.0000
  DOCUMENT_COUNT                 1.0000
  CONTACT_REACHABILITY           3.0000

Correlation with TARGET:
  AGE_YEARS                      -0.0782  ↓ higher = less risk
  YEARS_EMPLOYED                 -0.0750  ↓ higher = less risk
  CREDIT_INCOME_RATIO            -0.0077  ↓ higher = less risk
  ANNUITY_INCOME_RATIO           +0.0143  ↑ higher = more risk
  CREDIT_TERM                    +0.0127  ↑ higher = more risk
  EXT_SOURCE_MEAN                -0.2221  ↓ higher = less risk
  EXT_SOURCE_WEIGHTED            -0.2228  ↓ hi

CONTACT_REACHABILITY is positive (+0.0208) — more contact channels = more risk. Counterintuitive until you think about it: perhaps high-risk borrowers who anticipate collection calls may proactively provide more contact info. Or it could be that people with more phones/emails have more complex financial lives.

CREDIT_INCOME_RATIO is nearly flat at -0.0077 — much weaker than expected for a debt-to-income ratio. This is because the raw linear correlation misses the non-linear relationship. WOE binning will capture it better.

EXT_SOURCE_WEIGHTED at -0.2228 is stronger than any individual EXT_SOURCE score (-0.179 max). The combination is more powerful than any single score alone.

In [3]:
print("=" * 60)
print("BUREAU AGGREGATION")
print("=" * 60)

start = datetime.now()
bureau = pd.read_csv('../data/raw/bureau.csv')
print(f"bureau loaded: {bureau.shape}")
print(f"Unique SK_ID_CURR in bureau: "
      f"{bureau['SK_ID_CURR'].nunique():,}")
print(f"Clients in app not in bureau: "
      f"{app['SK_ID_CURR'].nunique() - bureau['SK_ID_CURR'].nunique():,}")

# Key columns to understand
print(f"\nCREDIT_ACTIVE value counts:")
print(bureau['CREDIT_ACTIVE'].value_counts())

print(f"\nCREDIT_TYPE value counts (top 8):")
print(bureau['CREDIT_TYPE'].value_counts().head(8))

BUREAU AGGREGATION
bureau loaded: (1716428, 17)
Unique SK_ID_CURR in bureau: 305,811
Clients in app not in bureau: 1,700

CREDIT_ACTIVE value counts:
CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

CREDIT_TYPE value counts (top 8):
CREDIT_TYPE
Consumer credit                  1251615
Credit card                       402195
Car loan                           27690
Mortgage                           18391
Microloan                          12413
Loan for business development       1975
Another type of loan                1017
Unknown type of loan                 555
Name: count, dtype: int64


In [4]:
# Aggregate bureau to one row per SK_ID_CURR
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    # Volume features
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('CREDIT_ACTIVE',
                         lambda x: (x == 'Active').sum()),
    BUREAU_CLOSED_COUNT=('CREDIT_ACTIVE',
                         lambda x: (x == 'Closed').sum()),
    BUREAU_BAD_DEBT_COUNT=('CREDIT_ACTIVE',
                           lambda x: (x == 'Bad debt').sum()),

    # Credit type flags
    BUREAU_CONSUMER_COUNT=('CREDIT_TYPE',
                           lambda x: (x == 'Consumer credit').sum()),
    BUREAU_CARD_COUNT=('CREDIT_TYPE',
                       lambda x: (x == 'Credit card').sum()),
    BUREAU_MORTGAGE_FLAG=('CREDIT_TYPE',
                          lambda x: int((x == 'Mortgage').sum() > 0)),

    # Delinquency — most important bureau features
    BUREAU_MAX_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_TOTAL_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUREAU_MEAN_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', 'mean'),

    # Debt burden
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_MEAN_DEBT=('AMT_CREDIT_SUM_DEBT', 'mean'),
    BUREAU_TOTAL_CREDIT=('AMT_CREDIT_SUM', 'sum'),

    # Credit history length
    BUREAU_MEAN_DAYS_CREDIT=('DAYS_CREDIT', 'mean'),
    BUREAU_MIN_DAYS_CREDIT=('DAYS_CREDIT', 'min'),
    BUREAU_MEAN_DAYS_ENDDATE=('DAYS_CREDIT_ENDDATE', 'mean'),

    # Credit update recency
    BUREAU_MEAN_DAYS_UPDATE=('DAYS_CREDIT_UPDATE', 'mean'),
).reset_index()

print(f"bureau_agg shape: {bureau_agg.shape}")
print(f"Expected: (305811, 16) — one row per client with bureau history")

# Join to main table
app = app.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(f"\napp shape after bureau join: {app.shape}")
print(f"Expected: (307511, {133 + len(bureau_agg.columns) - 1})")

# Check how many clients have no bureau history
no_bureau = app['BUREAU_LOAN_COUNT'].isna().sum()
print(f"\nClients with no bureau history: {no_bureau:,} "
      f"({no_bureau/len(app)*100:.2f}%)")

# Correlation of key bureau features with TARGET
bureau_features = [c for c in bureau_agg.columns
                   if c != 'SK_ID_CURR']
print(f"\nTop bureau feature correlations with TARGET:")
corrs = []
for f in bureau_features:
    corr = app[f].corr(app['TARGET'])
    corrs.append((f, corr))
corrs.sort(key=lambda x: abs(x[1]), reverse=True)
for f, corr in corrs[:10]:
    direction = "↑ risk" if corr > 0 else "↓ risk"
    print(f"  {f:<35} {corr:+.4f}  {direction}")

# Free memory — raw bureau no longer needed
del bureau
print(f"\nRaw bureau table deleted from memory.")

bureau_agg shape: (305811, 18)
Expected: (305811, 16) — one row per client with bureau history

app shape after bureau join: (307511, 150)
Expected: (307511, 150)

Clients with no bureau history: 44,020 (14.31%)

Top bureau feature correlations with TARGET:
  BUREAU_MEAN_DAYS_CREDIT             +0.0897  ↑ risk
  BUREAU_MIN_DAYS_CREDIT              +0.0752  ↑ risk
  BUREAU_MEAN_DAYS_UPDATE             +0.0689  ↑ risk
  BUREAU_ACTIVE_COUNT                 +0.0671  ↑ risk
  BUREAU_MEAN_DAYS_ENDDATE            +0.0470  ↑ risk
  BUREAU_CARD_COUNT                   +0.0348  ↑ risk
  BUREAU_CLOSED_COUNT                 -0.0308  ↓ risk
  BUREAU_MORTGAGE_FLAG                -0.0231  ↓ risk
  BUREAU_TOTAL_CREDIT                 -0.0141  ↓ risk
  BUREAU_TOTAL_OVERDUE                +0.0133  ↑ risk

Raw bureau table deleted from memory.


In [5]:
print("=" * 60)
print("PREVIOUS APPLICATION AGGREGATION")
print("=" * 60)

start = datetime.now()
prev = pd.read_csv('../data/raw/previous_application.csv')
print(f"previous_application loaded: {prev.shape}")
print(f"Unique SK_ID_CURR in prev: {prev['SK_ID_CURR'].nunique():,}")

print(f"\nNAME_CONTRACT_STATUS value counts:")
print(prev['NAME_CONTRACT_STATUS'].value_counts())

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS',
                         lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS',
                        lambda x: (x == 'Refused').sum()),
    PREV_CANCELED_COUNT=('NAME_CONTRACT_STATUS',
                         lambda x: (x == 'Canceled').sum()),
    PREV_APPROVAL_RATE=('NAME_CONTRACT_STATUS',
                        lambda x: (x == 'Approved').sum() / len(x)),
    PREV_MEAN_CREDIT=('AMT_CREDIT', 'mean'),
    PREV_MAX_CREDIT=('AMT_CREDIT', 'max'),
    PREV_MEAN_ANNUITY=('AMT_ANNUITY', 'mean'),
    PREV_MEAN_DOWNPAYMENT=('AMT_DOWN_PAYMENT', 'mean'),
    PREV_MEAN_DAYS_DECISION=('DAYS_DECISION', 'mean'),
    PREV_LAST_DAYS_DECISION=('DAYS_DECISION', 'max'),
    PREV_CONSUMER_COUNT=('NAME_CONTRACT_TYPE',
                         lambda x: (x == 'Consumer loans').sum()),
    PREV_CASH_COUNT=('NAME_CONTRACT_TYPE',
                     lambda x: (x == 'Cash loans').sum()),
    PREV_REVOLVING_COUNT=('NAME_CONTRACT_TYPE',
                          lambda x: (x == 'Revolving loans').sum()),
).reset_index()

print(f"\nprev_agg shape: {prev_agg.shape}")

app = app.merge(prev_agg, on='SK_ID_CURR', how='left')
print(f"app shape after prev join: {app.shape}")

no_prev = app['PREV_APP_COUNT'].isna().sum()
print(f"Clients with no prev application: {no_prev:,} "
      f"({no_prev/len(app)*100:.2f}%)")

print(f"\nTop prev feature correlations with TARGET:")
prev_features = [c for c in prev_agg.columns if c != 'SK_ID_CURR']
corrs = []
for f in prev_features:
    corr = app[f].corr(app['TARGET'])
    corrs.append((f, corr))
corrs.sort(key=lambda x: abs(x[1]), reverse=True)
for f, corr in corrs[:8]:
    direction = "↑ risk" if corr > 0 else "↓ risk"
    print(f"  {f:<35} {corr:+.4f}  {direction}")

del prev
print(f"\nRaw previous_application deleted from memory.")
print(f"Time: {datetime.now() - start}")

PREVIOUS APPLICATION AGGREGATION
previous_application loaded: (1670214, 37)
Unique SK_ID_CURR in prev: 338,857

NAME_CONTRACT_STATUS value counts:
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

prev_agg shape: (338857, 15)
app shape after prev join: (307511, 164)
Clients with no prev application: 16,454 (5.35%)

Top prev feature correlations with TARGET:
  PREV_REFUSED_COUNT                  +0.0645  ↑ risk
  PREV_APPROVAL_RATE                  -0.0635  ↓ risk
  PREV_MEAN_DAYS_DECISION             +0.0469  ↑ risk
  PREV_REVOLVING_COUNT                +0.0456  ↑ risk
  PREV_MEAN_ANNUITY                   -0.0349  ↓ risk
  PREV_APPROVED_COUNT                 -0.0316  ↓ risk
  PREV_MEAN_DOWNPAYMENT               -0.0246  ↓ risk
  PREV_CASH_COUNT                     +0.0227  ↑ risk

Raw previous_application deleted from memory.
Time: 0:04:16.336166


PREV_REFUSED_COUNT +0.0645 and PREV_APPROVAL_RATE -0.0635 are nearly mirror images — both saying the same thing from opposite directions. Prior refusals = higher current default risk. This makes intuitive sense: if other lenders already said no to this person, there's a reason.

PREV_REVOLVING_COUNT +0.0456 — more prior revolving loans = more risk. Revolving credit (credit cards) correlates with financial instability in this population.

PREV_MEAN_ANNUITY -0.0349 — higher prior loan payments = lower risk. Counterintuitive but consistent with income — people who took larger loans previously had more capacity to repay.

In [6]:
print("=" * 60)
print("INSTALLMENTS PAYMENTS AGGREGATION")
print("=" * 60)

start = datetime.now()
inst = pd.read_csv('../data/raw/installments_payments.csv')
print(f"installments_payments loaded: {inst.shape}")

# Engineer payment behavior features before aggregating
inst['PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['DAYS_LATE'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['DAYS_LATE'] = inst['DAYS_LATE'].clip(lower=0)
inst['DAYS_EARLY'] = inst['DAYS_INSTALMENT'] - inst['DAYS_ENTRY_PAYMENT']
inst['DAYS_EARLY'] = inst['DAYS_EARLY'].clip(lower=0)
inst['PAID_OVER'] = (inst['AMT_PAYMENT'] > inst['AMT_INSTALMENT']).astype(int)

inst_agg = inst.groupby('SK_ID_CURR').agg(
    INST_COUNT=('AMT_INSTALMENT', 'count'),
    INST_MEAN_PAYMENT_DIFF=('PAYMENT_DIFF', 'mean'),
    INST_MAX_PAYMENT_DIFF=('PAYMENT_DIFF', 'max'),
    INST_TOTAL_PAYMENT_DIFF=('PAYMENT_DIFF', 'sum'),
    INST_MEAN_DAYS_LATE=('DAYS_LATE', 'mean'),
    INST_MAX_DAYS_LATE=('DAYS_LATE', 'max'),
    INST_LATE_COUNT=('DAYS_LATE', lambda x: (x > 0).sum()),
    INST_LATE_RATE=('DAYS_LATE', lambda x: (x > 0).mean()),
    INST_MEAN_DAYS_EARLY=('DAYS_EARLY', 'mean'),
    INST_OVERPAY_COUNT=('PAID_OVER', 'sum'),
).reset_index()

print(f"\ninst_agg shape: {inst_agg.shape}")

app = app.merge(inst_agg, on='SK_ID_CURR', how='left')
print(f"app shape after inst join: {app.shape}")

no_inst = app['INST_COUNT'].isna().sum()
print(f"Clients with no installment history: {no_inst:,} "
      f"({no_inst/len(app)*100:.2f}%)")

print(f"\nTop installment feature correlations with TARGET:")
inst_features = [c for c in inst_agg.columns if c != 'SK_ID_CURR']
corrs = []
for f in inst_features:
    corr = app[f].corr(app['TARGET'])
    corrs.append((f, corr))
corrs.sort(key=lambda x: abs(x[1]), reverse=True)
for f, corr in corrs:
    direction = "↑ risk" if corr > 0 else "↓ risk"
    print(f"  {f:<35} {corr:+.4f}  {direction}")

del inst
print(f"\nRaw installments_payments deleted from memory.")
print(f"Time: {datetime.now() - start}")

INSTALLMENTS PAYMENTS AGGREGATION
installments_payments loaded: (13605401, 8)

inst_agg shape: (339587, 11)
app shape after inst join: (307511, 174)
Clients with no installment history: 15,868 (5.16%)

Top installment feature correlations with TARGET:
  INST_LATE_RATE                      +0.0700  ↑ risk
  INST_LATE_COUNT                     +0.0304  ↑ risk
  INST_MEAN_PAYMENT_DIFF              +0.0293  ↑ risk
  INST_TOTAL_PAYMENT_DIFF             +0.0273  ↑ risk
  INST_OVERPAY_COUNT                  -0.0272  ↓ risk
  INST_MEAN_DAYS_EARLY                -0.0211  ↓ risk
  INST_COUNT                          -0.0211  ↓ risk
  INST_MAX_PAYMENT_DIFF               +0.0191  ↑ risk
  INST_MEAN_DAYS_LATE                 +0.0089  ↑ risk
  INST_MAX_DAYS_LATE                  +0.0038  ↑ risk

Raw installments_payments deleted from memory.
Time: 0:00:44.352064



**Source:** `installments_payments` — 13,605,401 rows → aggregated to 339,587 clients → joined to 307,511 training rows  
**New features added:** 10 | **Running feature total:** 174 columns  
**Clients with no installment history:** 15,868 (5.16%) — will bin as "no history" in WOE

### What This Table Captures
Installment payments tracks whether clients paid their previous Home Credit loans on time and in full. Each row is one scheduled payment — we aggregated across all payments per client to get behavioral signals about payment discipline.

### Key Findings

| Feature | Correlation | Interpretation |
|---|---|---|
| `INST_LATE_RATE` | +0.0700 ↑ risk | Strongest signal — % of payments made late. Normalized for loan volume, so more reliable than raw counts. |
| `INST_LATE_COUNT` | +0.0304 ↑ risk | Raw count of late payments. Weaker than rate but adds volume context. |
| `INST_MEAN_PAYMENT_DIFF` | +0.0293 ↑ risk | On average, clients paid less than required. Chronic underpayment pattern. |
| `INST_OVERPAY_COUNT` | −0.0272 ↓ risk | Clients who overpay are lower risk — signals financial cushion and good intent. |
| `INST_COUNT` | −0.0211 ↓ risk | More installment loans = lower risk. Likely a proxy for established credit history. |
| `INST_MAX_DAYS_LATE` | +0.0038 ↑ risk | Weakest feature — max values are noisy and skewed by single outlier events. Likely to be dropped in WOE selection. |

### Project Impact
- `INST_LATE_RATE` is the clear keeper and moves to **Tier 4** of the feature shortlist
- `INST_MAX_DAYS_LATE` and `INST_MAX_PAYMENT_DIFF` are candidates for **dropping** — max-based features rarely bin cleanly in WOE/IV selection
- The 5.16% thin-file rate is consistent with prior application data (5.35%), suggesting the same population of new-to-credit clients appears across both tables

In [9]:
import time
import datetime
start = time.time()

print("=" * 60)
print("CREDIT CARD BALANCE AGGREGATION")
print("=" * 60)

cc = pd.read_csv('../data/raw/credit_card_balance.csv')
print(f"credit_card_balance loaded: {cc.shape}")


# True total drawings across all types (ATM, POS, other, general)
cc['AMT_DRAWINGS_TOTAL'] = (
    cc['AMT_DRAWINGS_ATM_CURRENT'].fillna(0) +
    cc['AMT_DRAWINGS_CURRENT'].fillna(0) +
    cc['AMT_DRAWINGS_OTHER_CURRENT'].fillna(0) +
    cc['AMT_DRAWINGS_POS_CURRENT'].fillna(0)
)

# How much of the credit limit is being used (utilization)
cc['CC_UTILIZATION'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)

# Minimum payment flag — did they only pay the minimum?
cc['CC_MIN_PAYMENT_FLAG'] = (
    cc['AMT_PAYMENT_CURRENT'] <= cc['AMT_INST_MIN_REGULARITY'] + 1
).astype(int)

# Payment ratio — what fraction of the balance did they pay off
cc['CC_PAYMENT_RATIO'] = cc['AMT_PAYMENT_CURRENT'] / (cc['AMT_BALANCE'] + 1)

# Aggregate per client 
cc_agg = cc.groupby('SK_ID_CURR').agg(
    CC_COUNT=('SK_ID_PREV', 'nunique'),                        # number of credit cards
    CC_MEAN_UTILIZATION=('CC_UTILIZATION', 'mean'),            # avg utilization rate
    CC_MAX_UTILIZATION=('CC_UTILIZATION', 'max'),              # peak utilization
    CC_MEAN_BALANCE=('AMT_BALANCE', 'mean'),                   # avg balance carried
    CC_MIN_PAYMENT_RATE=('CC_MIN_PAYMENT_FLAG', 'mean'),       # % months only min paid
    CC_MEAN_PAYMENT_RATIO=('CC_PAYMENT_RATIO', 'mean'),        # avg paydown rate
    CC_TOTAL_DRAWINGS=('AMT_DRAWINGS_TOTAL', 'sum'),           # total cash/card drawings
    CC_MONTHS_ACTIVE=('MONTHS_BALANCE', 'count'),              # months of cc history
).reset_index()

print(f"cc_agg shape: {cc_agg.shape}")

#Join to main app 
app = app.merge(cc_agg, on='SK_ID_CURR', how='left')
print(f"app shape after cc join: {app.shape}")
print(f"Clients with no credit card history: {app['CC_COUNT'].isna().sum():,} ({app['CC_COUNT'].isna().mean()*100:.2f}%)")

#Correlations
print("\nTop credit card feature correlations with TARGET:")
cc_cols = [c for c in app.columns if c.startswith('CC_')]
corrs = app[cc_cols + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values(key=abs, ascending=False)
for feat, val in corrs.items():
    direction = "↑ risk" if val > 0 else "↓ risk"
    print(f"  {feat:<35} {val:+.4f}  {direction}")

del cc, cc_agg
print("\nRaw credit_card_balance deleted from memory.")
print(f"Time: {datetime.timedelta(seconds=time.time()-start)}")

CREDIT CARD BALANCE AGGREGATION
credit_card_balance loaded: (3840312, 23)
cc_agg shape: (103558, 9)
app shape after cc join: (307511, 182)
Clients with no credit card history: 220,606 (71.74%)

Top credit card feature correlations with TARGET:
  CC_MEAN_BALANCE                     +0.0872  ↑ risk
  CC_MONTHS_ACTIVE                    -0.0605  ↓ risk
  CC_TOTAL_DRAWINGS                   +0.0238  ↑ risk
  CC_MIN_PAYMENT_RATE                 +0.0085  ↑ risk
  CC_MEAN_PAYMENT_RATIO               -0.0085  ↓ risk
  CC_COUNT                            +0.0044  ↑ risk
  CC_MEAN_UTILIZATION                 -0.0019  ↓ risk
  CC_MAX_UTILIZATION                  -0.0015  ↓ risk

Raw credit_card_balance deleted from memory.
Time: 0:00:06.491470


In [ ]:
import time
import datetime
start = time.time()

print("=" * 60)
print("POS_CASH_BALANCE AGGREGATION")
print("=" * 60)

pos = pd.read_csv('../data/raw/POS_CASH_balance.csv')
print(f"POS_CASH_balance loaded: {pos.shape}")

#Engineer before aggregating 
# Was the client ever late on this POS/cash loan?
pos['POS_LATE_FLAG'] = (pos['SK_DPD'] > 0).astype(int)

# Was it seriously delinquent (>30 days)?
pos['POS_SERIOUS_DPD_FLAG'] = (pos['SK_DPD_DEF'] > 0).astype(int)

# How far through the loan term are they?
pos['POS_TERM_PROGRESS'] = pos['CNT_INSTALMENT_FUTURE'] / (pos['CNT_INSTALMENT'] + 1)

#Aggregate per client 
pos_agg = pos.groupby('SK_ID_CURR').agg(
    POS_COUNT=('SK_ID_PREV', 'nunique'),                        # number of POS/cash loans
    POS_MONTHS_ACTIVE=('MONTHS_BALANCE', 'count'),              # total months of history
    POS_LATE_RATE=('POS_LATE_FLAG', 'mean'),                    # % months with any DPD
    POS_SERIOUS_DPD_RATE=('POS_SERIOUS_DPD_FLAG', 'mean'),      # % months with serious DPD
    POS_MAX_DPD=('SK_DPD', 'max'),                              # worst DPD ever
    POS_MEAN_DPD=('SK_DPD', 'mean'),                            # avg DPD across months
    POS_COMPLETED_COUNT=('NAME_CONTRACT_STATUS', 
                         lambda x: (x == 'Completed').sum()),   # loans fully paid off
    POS_ACTIVE_COUNT=('NAME_CONTRACT_STATUS',
                      lambda x: (x == 'Active').sum()),         # currently active loans
).reset_index()

# Completion rate — paid off loans / total loans
pos_agg['POS_COMPLETION_RATE'] = (
    pos_agg['POS_COMPLETED_COUNT'] / (pos_agg['POS_COUNT'] + 1)
)

print(f"pos_agg shape: {pos_agg.shape}")

#Join to main app
app = app.merge(pos_agg, on='SK_ID_CURR', how='left')
print(f"app shape after POS join: {app.shape}")
print(f"Clients with no POS/cash history: {app['POS_COUNT'].isna().sum():,} ({app['POS_COUNT'].isna().mean()*100:.2f}%)")

#Correlations
print("\nTop POS_CASH feature correlations with TARGET:")
pos_cols = [c for c in app.columns if c.startswith('POS_')]
corrs = app[pos_cols + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values(key=abs, ascending=False)
for feat, val in corrs.items():
    direction = "↑ risk" if val > 0 else "↓ risk"
    print(f"  {feat:<35} {val:+.4f}  {direction}")

del pos, pos_agg
print("\nRaw POS_CASH_balance deleted from memory.")
print(f"Time: {datetime.timedelta(seconds=time.time()-start)}")

POS_CASH_BALANCE AGGREGATION
POS_CASH_balance loaded: (10001358, 8)
